# 🔗 ImageBind — CSC14005 Final Project
**Topic 17 | Nhóm 2 người | Focused type**

| Tuần | Nội dung |
|------|----------|
| Tuần 1 | Setup môi trường · Clone repo · Chạy demo · Chuẩn bị data |
| Tuần 2 | **Training loop (FIXED)** · Freeze backbone · Checkpoint · Evaluate · Metric |
| Tuần 3 | Benchmark hyperparams · Full evaluate · Vẽ biểu đồ |

> **Chạy trên Kaggle**: Accelerator → GPU T4 · Internet → ON

---
# 📗 TUẦN 1 — Setup & Chuẩn bị

## 1.1 Cài đặt môi trường

In [8]:
import os

if not os.path.exists('ImageBind'):
    !git clone https://github.com/facebookresearch/ImageBind.git
    print('Cloned!')
else:
    print('Repo da ton tai, bo qua.')

%cd ImageBind

!pip install -q .
!pip install -q timm ftfy regex einops fvcore tqdm matplotlib seaborn pandas
print('\n=== Cai dat xong ===')

Cloning into 'ImageBind'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 187 (delta 84), reused 48 (delta 48), pack-reused 67 (from 2)
Receiving objects: 100% (187/187), 2.65 MiB | 15.52 MiB/s, done.
Resolving deltas: 100% (92/92), done.
Cloned!
/kaggle/working/ImageBind/ImageBind
  Preparing metadata (setup.py) ... done

=== Cai dat xong ===


In [9]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import torch

print('=== Kiem tra moi truong ===')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nSu dung device  : {DEVICE}')

=== Kiem tra moi truong ===
PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB

Su dung device  : cuda


## 1.2 Tải pretrained weights

In [10]:
import os

os.makedirs('.checkpoints', exist_ok=True)
WEIGHT_PATH = '/kaggle/input/datasets/hongngctng/checkpoint1/imagebind_huge.pth'

if not os.path.exists(WEIGHT_PATH):
    print('Dang tai pretrained weights (~4.5GB)...')
    !wget -q --show-progress -O {WEIGHT_PATH} \
        https://dl.fbaipublicfiles.com/imagebind/imagebind_huge.pth
    print('Tai xong!')
else:
    size_mb = os.path.getsize(WEIGHT_PATH) / 1e6
    print(f'Weights da co san: {size_mb:.0f} MB')

Weights da co san: 4804 MB


## 1.3 Load model & Smoke test

In [11]:
from imagebind import data as ibdata
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType

print('Dang load model...')
model = imagebind_model.imagebind_huge(pretrained=True)
model.eval()
model.to(DEVICE)
print('Load model thanh cong!')

total_params = sum(p.numel() for p in model.parameters())
print(f'Tong params: {total_params / 1e6:.1f}M')
print('(Sau khi freeze backbone o Tuan 2, chi con ~vài MB trainable)')

Dang load model...


100%|██████████| 4.47G/4.47G [00:13<00:00, 353MB/s]


Load model thanh cong!
Tong params: 1200.8M
(Sau khi freeze backbone o Tuan 2, chi con ~vài MB trainable)


In [12]:
import torch

texts = ['A dog is barking', 'Fire crackling sound',
         'Ocean waves crashing', 'Rain falling on roof']

inputs = {ModalityType.TEXT: ibdata.load_and_transform_text(texts, DEVICE)}

with torch.no_grad():
    embeddings = model(inputs)

emb = embeddings[ModalityType.TEXT]
print(f'Text embedding shape: {emb.shape}')  # [4, 1024]
print(f'Embedding norm      : {emb.norm(dim=-1)}')  # ~1.0
print('Smoke test OK!')

Text embedding shape: torch.Size([4, 1024])
Embedding norm      : tensor([100.0000, 100.0000, 100.0000, 100.0000], device='cuda:0')
Smoke test OK!


---
# 📘 TUẦN 2 — Training Loop (FIXED)

**Các fix so với lần trước:**
| # | Vấn đề cũ | Fix |
|---|---|---|
| 1 | Train toàn bộ 1.2B params | ✅ Freeze backbone, chỉ train head |
| 2 | 2 epoch → acc=5% | ✅ 30 epoch với OneCycleLR |
| 3 | Không resume được | ✅ Load checkpoint đúng cách |
| 4 | Không có label smoothing | ✅ CrossEntropyLoss(label_smoothing=0.1) |
| 5 | Không clip gradient | ✅ clip_grad_norm_(max=1.0) |

## 2.1 Dataset — ESC-50

In [ ]:

import os, json, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchaudio
from torchvision import transforms

# ESC-50 paths on Kaggle.
ESC50_ROOT  = "/kaggle/input/datasets/hongngctng/dataesc50/ESC-50-master"
AUDIO_DIR   = os.path.join(ESC50_ROOT, "audio")
META_PATH   = os.path.join(ESC50_ROOT, "meta", "esc50.csv")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

AUDIO_SAMPLE_RATE = 16000
AUDIO_CLIP_SECONDS = 2
AUDIO_TARGET_LENGTH = 204
AUDIO_NUM_MEL_BINS = 128
RANDOM_BASELINE = 1.0 / 50

meta = pd.read_csv(META_PATH)
all_classes = sorted(meta["category"].unique())
class2idx = {name: idx for idx, name in enumerate(all_classes)}
idx2class = {idx: name for name, idx in class2idx.items()}
NUM_CLASSES = len(all_classes)

train_meta = meta[meta["fold"].isin([1, 2, 3])].copy()
val_meta   = meta[meta["fold"] == 4].copy()
test_meta  = meta[meta["fold"] == 5].copy()


class ESC50PathDataset(Dataset):
    """ESC-50 dataset that returns wav path and a stable global label id."""
    def __init__(self, meta_df, audio_dir, class2idx):
        self.meta = meta_df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.class2idx = dict(class2idx)
        self.idx2class = {idx: name for name, idx in self.class2idx.items()}
        self.num_classes = len(self.class2idx)

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        row = self.meta.iloc[idx]
        wav_path = os.path.join(self.audio_dir, row["filename"])
        label = self.class2idx[row["category"]]
        return wav_path, label


train_dataset = ESC50PathDataset(train_meta, AUDIO_DIR, class2idx)
val_dataset   = ESC50PathDataset(val_meta,   AUDIO_DIR, class2idx)
test_dataset  = ESC50PathDataset(test_meta,  AUDIO_DIR, class2idx)

assert train_dataset.class2idx == val_dataset.class2idx == test_dataset.class2idx
assert NUM_CLASSES == 50, f"Expected 50 ESC-50 classes, got {NUM_CLASSES}"

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Num classes: {NUM_CLASSES}")
print(f"Random baseline Top-1: {RANDOM_BASELINE * 100:.2f}%")
print("Class mapping identical across train/val/test:", train_dataset.class2idx == val_dataset.class2idx == test_dataset.class2idx)
print("Classes:", all_classes[:5], "...")


In [ ]:

EXTRACT_BATCH_SIZE = 16
TRAIN_BATCH_SIZE = 128
NUM_WORKERS = 2

path_train_loader = DataLoader(
    train_dataset, batch_size=EXTRACT_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False,
)
path_val_loader = DataLoader(
    val_dataset, batch_size=EXTRACT_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False,
)
path_test_loader = DataLoader(
    test_dataset, batch_size=EXTRACT_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False,
)


def fallback_load_and_transform_audio_data(
    audio_paths,
    device,
    num_mel_bins=AUDIO_NUM_MEL_BINS,
    target_length=AUDIO_TARGET_LENGTH,
    sample_rate=AUDIO_SAMPLE_RATE,
    clip_duration=AUDIO_CLIP_SECONDS,
    mean=-4.268,
    std=9.138,
):
    """ImageBind-compatible fallback based on its official kaldi fbank pipeline."""
    audio_outputs = []
    target_samples = int(sample_rate * clip_duration)

    for audio_path in audio_paths:
        waveform, sr = torchaudio.load(audio_path)
        if sr != sample_rate:
            waveform = torchaudio.functional.resample(waveform, orig_freq=sr, new_freq=sample_rate)
        waveform = waveform.mean(dim=0, keepdim=True)

        if waveform.shape[1] < target_samples:
            pad = target_samples - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad))
        else:
            waveform = waveform[:, :target_samples]

        waveform = waveform - waveform.mean()
        fbank = torchaudio.compliance.kaldi.fbank(
            waveform,
            htk_compat=True,
            sample_frequency=sample_rate,
            use_energy=False,
            window_type="hanning",
            num_mel_bins=num_mel_bins,
            dither=0.0,
            frame_length=25,
            frame_shift=10,
        )
        fbank = fbank.transpose(0, 1)
        frame_pad = target_length - fbank.size(1)
        if frame_pad > 0:
            fbank = torch.nn.functional.pad(fbank, (0, frame_pad), mode="constant", value=0)
        elif frame_pad < 0:
            fbank = fbank[:, :target_length]

        fbank = fbank.unsqueeze(0)
        fbank = transforms.Normalize(mean=mean, std=std)(fbank)
        audio_outputs.append(fbank.unsqueeze(0))

    return torch.stack(audio_outputs, dim=0).to(device)


def make_audio_inputs(audio_paths, device):
    """Prefer official ImageBind preprocessing; keep a deterministic fallback."""
    audio_paths = list(audio_paths)
    try:
        return ibdata.load_and_transform_audio_data(audio_paths, device)
    except Exception as exc:
        print(f"[WARN] Official ImageBind audio preprocessing failed: {exc}")
        print("[WARN] Falling back to local ImageBind-compatible kaldi fbank pipeline.")
        return fallback_load_and_transform_audio_data(audio_paths, device)


sample_paths, sample_labels = next(iter(path_train_loader))
sample_audio = make_audio_inputs(sample_paths[:2], DEVICE)
print(f"Path batches: train={len(path_train_loader)} | val={len(path_val_loader)} | test={len(path_test_loader)}")
print(f"Audio input shape: {tuple(sample_audio.shape)}")
print(f"Expected ImageBind audio shape: (B, 3, 1, 128, 204) with official preprocessing")
print(f"Sample labels: {sample_labels[:5]}")


## 2.2 Freeze backbone — FIX #1

> **Mục đích**: Giữ nguyên pretrained weights của ImageBind, chỉ train audio head (~vài MB thay vì 1.2GB). VRAM từ ~15GB → ~4GB.

In [ ]:

def freeze_backbone(backbone):
    """Freeze the full pretrained ImageBind backbone and keep it in eval mode."""
    backbone.eval()
    for param in backbone.parameters():
        param.requires_grad = False

    total = sum(p.numel() for p in backbone.parameters())
    trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    print("[Freeze] pretrained ImageBind backbone")
    print(f"  Trainable backbone params: {trainable:,} / {total:,}")
    return backbone


FREEZE_MODE = "frozen_embedding_extractor"
model = freeze_backbone(model)
assert not any(p.requires_grad for p in model.parameters())


## 2.3 AudioClassifier — head gắn lên embedding

In [ ]:

import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm


@torch.no_grad()
def extract_audio_embeddings(backbone, loader, device, split_name):
    backbone.eval()
    embeddings, labels = [], []

    for audio_paths, label_batch in tqdm(loader, desc=f"Extract {split_name}"):
        audio_input = make_audio_inputs(audio_paths, device)
        emb = backbone({ModalityType.AUDIO: audio_input})[ModalityType.AUDIO]
        embeddings.append(emb.detach().cpu())
        labels.append(label_batch.long().cpu())

    embeddings = torch.cat(embeddings, dim=0).float()
    labels = torch.cat(labels, dim=0).long()
    norms = embeddings.norm(dim=1)
    print(
        f"{split_name:5s} embeddings: shape={tuple(embeddings.shape)} | "
        f"norm mean={norms.mean():.2f} min={norms.min():.2f} max={norms.max():.2f}"
    )
    return embeddings, labels


train_embeddings, train_labels = extract_audio_embeddings(model, path_train_loader, DEVICE, "train")
val_embeddings, val_labels = extract_audio_embeddings(model, path_val_loader, DEVICE, "val")
test_embeddings, test_labels = extract_audio_embeddings(model, path_test_loader, DEVICE, "test")

EMBED_DIM = train_embeddings.shape[1]
assert EMBED_DIM == 1024, f"Expected ImageBind embedding dim 1024, got {EMBED_DIM}"
assert train_labels.min().item() >= 0 and train_labels.max().item() < NUM_CLASSES

train_loader = DataLoader(
    TensorDataset(train_embeddings, train_labels),
    batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=0,
)
val_loader = DataLoader(
    TensorDataset(val_embeddings, val_labels),
    batch_size=TRAIN_BATCH_SIZE, shuffle=False, num_workers=0,
)
test_loader = DataLoader(
    TensorDataset(test_embeddings, test_labels),
    batch_size=TRAIN_BATCH_SIZE, shuffle=False, num_workers=0,
)


class AudioEmbeddingClassifier(nn.Module):
    """Small classifier trained on frozen ImageBind audio embeddings."""
    def __init__(self, embed_dim=1024, num_classes=50):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.2),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, embeddings):
        return self.classifier(embeddings)


train_model = AudioEmbeddingClassifier(
    embed_dim=EMBED_DIM,
    num_classes=NUM_CLASSES,
).to(DEVICE)

with torch.no_grad():
    logits = train_model(train_embeddings[:4].to(DEVICE))

print(f"Classifier input embedding shape: {tuple(train_embeddings[:4].shape)}")
print(f"Classifier output shape: {tuple(logits.shape)}")
print(f"Backbone still eval: {not model.training}")
print(f"Backbone trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")


## 2.4 Optimizer, Scheduler, Loss — FIX #2 #3

In [ ]:

from torch.optim.lr_scheduler import OneCycleLR

NUM_EPOCHS = 60
LR_MAX = 3e-3

optimizer = torch.optim.AdamW(
    train_model.parameters(),
    lr=LR_MAX,
    weight_decay=1e-3,
)

scheduler = OneCycleLR(
    optimizer,
    max_lr=LR_MAX,
    epochs=NUM_EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.15,
    anneal_strategy="cos",
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

print(f"Optimizer : AdamW | LR_max={LR_MAX}")
print(f"Scheduler : OneCycleLR | {NUM_EPOCHS} epochs")
print("Loss      : CrossEntropyLoss(label_smoothing=0.05)")


## 2.5 Checkpoint — Resume & Save — FIX #4

In [ ]:

import os, json

CKPT_DIR = "checkpoints"
BEST_CLASSIFIER_PATH = os.path.join(CKPT_DIR, "best_classifier.pth")
RESUME = None
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs("results", exist_ok=True)


def save_checkpoint(model, optimizer, scheduler, epoch, best_acc, history, path=BEST_CLASSIFIER_PATH):
    torch.save({
        "epoch": epoch,
        "best_acc": best_acc,
        "classifier_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
        "history": history,
        "class2idx": class2idx,
        "idx2class": idx2class,
        "embed_dim": EMBED_DIM,
        "freeze_mode": FREEZE_MODE,
    }, path)
    print(f"  Saved: {path}")


def load_checkpoint(model, optimizer, scheduler, path):
    if path and os.path.exists(path):
        ckpt = torch.load(path, map_location=DEVICE)
        model.load_state_dict(ckpt["classifier_state_dict"])

        if optimizer and "optimizer_state_dict" in ckpt:
            try:
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            except Exception as exc:
                print(f"  [WARN] Optimizer load failed, reset optimizer: {exc}")

        if scheduler and ckpt.get("scheduler_state_dict"):
            try:
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            except Exception as exc:
                print(f"  [WARN] Scheduler load failed, reset scheduler: {exc}")

        start_epoch = ckpt.get("epoch", 0) + 1
        best_acc = ckpt.get("best_acc", 0.0)
        history = ckpt.get("history", [])
        print(f"[Resume] epoch={start_epoch} | best_acc={best_acc:.2f}%")
        return start_epoch, best_acc, history

    print("[Start] No classifier checkpoint; train from scratch")
    return 0, 0.0, []


start_epoch, best_acc, history = load_checkpoint(train_model, optimizer, scheduler, RESUME)


## 2.6 Training Loop chính — FIX #5 (gradient clipping)

In [ ]:

import time


def run_epoch(classifier, loader, criterion, optimizer, scheduler, device, is_train):
    model.eval()
    classifier.train(is_train)
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc="Train" if is_train else "Val  ", leave=False)
    for emb_batch, label_batch in pbar:
        embeddings = emb_batch.to(device, non_blocking=True)
        labels = label_batch.to(device, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            logits = classifier(embeddings)
            loss = criterion(logits, labels)

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(classifier.parameters(), max_norm=1.0)
            optimizer.step()
            if scheduler:
                scheduler.step()

        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item() * labels.size(0)

        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct / total * 100:.1f}%")

    return total_loss / total, correct / total * 100


print("=" * 58)
print(f"  Linear probe on frozen ImageBind embeddings | epochs={NUM_EPOCHS} | device={DEVICE}")
print("=" * 58)
print(f"Random baseline Top-1: {RANDOM_BASELINE * 100:.2f}%\n")

for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()

    train_loss, train_acc = run_epoch(
        train_model, train_loader, criterion, optimizer, scheduler, DEVICE, is_train=True,
    )
    val_loss, val_acc = run_epoch(
        train_model, val_loader, criterion, None, None, DEVICE, is_train=False,
    )

    elapsed = time.time() - t0
    current_lr = optimizer.param_groups[0]["lr"]

    log = {
        "epoch": epoch,
        "train_loss": round(train_loss, 4),
        "train_acc": round(train_acc, 2),
        "val_loss": round(val_loss, 4),
        "val_acc": round(val_acc, 2),
        "lr": round(current_lr, 8),
        "time_s": round(elapsed, 1),
    }
    history.append(log)

    print(
        f"Epoch [{epoch + 1:02d}/{NUM_EPOCHS}] "
        f"| Train loss={train_loss:.4f} acc={train_acc:.1f}% "
        f"| Val loss={val_loss:.4f} acc={val_acc:.1f}% "
        f"| LR={current_lr:.2e} | {elapsed:.0f}s"
    )

    if val_acc > best_acc:
        best_acc = val_acc
        save_checkpoint(train_model, optimizer, scheduler, epoch, best_acc, history)
        print(f"  New best val acc: {best_acc:.2f}%")

with open("results/train_history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"\nTraining done. Best val acc: {best_acc:.2f}%")
print(f"Best classifier checkpoint: {BEST_CLASSIFIER_PATH}")
print(f"Backbone trainable params after training: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")


## 2.7 Learning curve

In [ ]:
import matplotlib.pyplot as plt

epochs_x   = [h['epoch'] + 1 for h in history]
train_acc  = [h['train_acc']  for h in history]
val_acc    = [h['val_acc']    for h in history]
train_loss = [h['train_loss'] for h in history]
val_loss   = [h['val_loss']   for h in history]
lr_hist    = [h['lr']         for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'ImageBind Fine-tuning — freeze={FREEZE_MODE}', fontsize=12, fontweight='bold')

# Accuracy
axes[0].plot(epochs_x, train_acc, label='Train', marker='o', markersize=3)
axes[0].plot(epochs_x, val_acc,   label='Val',   marker='s', markersize=3)
axes[0].axhline(best_acc, color='red', linestyle='--', alpha=0.5, label=f'Best={best_acc:.1f}%')
axes[0].set(xlabel='Epoch', ylabel='Accuracy (%)', title='Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(epochs_x, train_loss, label='Train', marker='o', markersize=3)
axes[1].plot(epochs_x, val_loss,   label='Val',   marker='s', markersize=3)
axes[1].set(xlabel='Epoch', ylabel='Loss', title='Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

# LR schedule
axes[2].plot(epochs_x, lr_hist, color='darkorange', marker='.', markersize=3)
axes[2].set(xlabel='Epoch', ylabel='Learning Rate', title='LR (OneCycleLR)')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/learning_curve.png', dpi=120)
plt.show()
print('Luu: results/learning_curve.png')

## 2.8 Evaluate trên Test set

In [ ]:

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np

best_ckpt = torch.load(BEST_CLASSIFIER_PATH, map_location=DEVICE)
train_model.load_state_dict(best_ckpt["classifier_state_dict"])
best_acc = best_ckpt["best_acc"]
print(f"Loaded best classifier | epoch={best_ckpt['epoch'] + 1} | best_val_acc={best_acc:.2f}%")

model.eval()
train_model.eval()
all_preds, all_labels, all_logits = [], [], []

with torch.no_grad():
    for emb_batch, label_batch in tqdm(test_loader, desc="Test eval"):
        logits = train_model(emb_batch.to(DEVICE))
        preds = logits.argmax(dim=-1).cpu()
        all_logits.append(logits.cpu())
        all_preds.extend(preds.numpy())
        all_labels.extend(label_batch.numpy())

all_logits = torch.cat(all_logits, dim=0)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
test_acc = (all_preds == all_labels).mean() * 100

print(f"\n=== Test Top-1 Accuracy: {test_acc:.2f}% ===")
print(f"Random baseline Top-1: {RANDOM_BASELINE * 100:.2f}%\n")

class_names = [test_dataset.idx2class[i] for i in range(NUM_CLASSES)]
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))


In [ ]:
import matplotlib.pyplot as plt

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(18, 15))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', linewidths=0.3,
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True',      fontsize=11)
ax.set_title(f'Confusion Matrix - Test Acc={test_acc:.1f}%', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=120)
plt.show()
print('Saved: results/confusion_matrix.png')


In [ ]:

top5 = all_logits.topk(5, dim=-1).indices
labels_for_top5 = torch.tensor(all_labels).unsqueeze(1).expand_as(top5)
top5_acc = (top5 == labels_for_top5).any(dim=1).float().mean().item() * 100

print(f"Top-1 Accuracy : {test_acc:.2f}%")
print(f"Top-5 Accuracy : {top5_acc:.2f}%")
print(f"Random Top-1   : {RANDOM_BASELINE * 100:.2f}%")

results_summary = {
    "method": "frozen_imagebind_audio_embeddings_linear_classifier",
    "freeze_mode": FREEZE_MODE,
    "num_epochs": NUM_EPOCHS,
    "num_classes": NUM_CLASSES,
    "random_baseline_top1": round(RANDOM_BASELINE * 100, 2),
    "best_val_acc": round(best_acc, 2),
    "test_top1": round(test_acc, 2),
    "test_top5": round(top5_acc, 2),
    "checkpoint": BEST_CLASSIFIER_PATH,
    "class2idx": class2idx,
    "history": history,
}
with open("results/final_results.json", "w") as f:
    json.dump(results_summary, f, indent=2)
print("Saved: results/final_results.json")


---
# 📙 TUẦN 3 — Benchmark & Cross-modal Demo

## 3.1 Zero-shot Text → Audio Retrieval

In [ ]:

@torch.no_grad()
def demo_text_to_audio(backbone, query_texts, test_embeddings, test_labels, test_dataset, device, top_k=5):
    backbone.eval()
    text_in = {ModalityType.TEXT: ibdata.load_and_transform_text(query_texts, device)}
    text_emb = F.normalize(backbone(text_in)[ModalityType.TEXT], dim=-1).cpu()
    audio_emb = F.normalize(test_embeddings, dim=-1).cpu()
    audio_labels = test_labels.cpu().numpy()

    sim = text_emb @ audio_emb.T
    results = []
    for q_idx, q_text in enumerate(query_texts):
        topk_idx = sim[q_idx].topk(top_k).indices.numpy()
        topk_labels = [test_dataset.idx2class[int(audio_labels[i])] for i in topk_idx]
        topk_scores = sim[q_idx][topk_idx].numpy()
        results.append({"query": q_text, "top_classes": topk_labels, "top_scores": topk_scores.tolist()})
    return results


TEXT_QUERIES = [
    "sound of a dog barking outside",
    "rain falling on a rooftop",
    "crackling fire in a fireplace",
    "ocean waves on a beach",
    "a baby crying loudly",
]

print("=== Demo: Text -> Audio Retrieval (Top-5) ===")
t2a_results = demo_text_to_audio(
    backbone=model,
    query_texts=TEXT_QUERIES,
    test_embeddings=test_embeddings,
    test_labels=test_labels,
    test_dataset=test_dataset,
    device=DEVICE,
    top_k=5,
)

for r in t2a_results:
    print(f"\nQuery: '{r['query']}'")
    for rank, (cls, sc) in enumerate(zip(r["top_classes"], r["top_scores"]), 1):
        print(f"  #{rank}: {cls:<25s} (score={sc:.4f})")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_queries = len(t2a_results)
fig, axes = plt.subplots(1, n_queries, figsize=(4 * n_queries, 5))
fig.suptitle('Text → Audio Retrieval (Top-5 per Query)', fontsize=12, fontweight='bold')

for ax, res in zip(axes, t2a_results):
    classes = res['top_classes']
    scores  = res['top_scores']
    ax.barh(range(len(classes))[::-1], scores, color='steelblue', alpha=0.85)
    ax.set_yticks(range(len(classes))[::-1])
    ax.set_yticklabels(classes, fontsize=8)
    ax.set_xlabel('Cosine Score')
    q_short = res['query'][:35] + '...' if len(res['query']) > 35 else res['query']
    ax.set_title(q_short, fontsize=8)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('results/demo_text_to_audio.png', dpi=120)
plt.show()
print('Luu: results/demo_text_to_audio.png')

## 3.2 Recall@K — Cross-modal

In [ ]:

@torch.no_grad()
def compute_recall_at_k(backbone, test_dataset, test_embeddings, test_labels, device, ks=(1, 5, 10)):
    backbone.eval()
    class_names = [test_dataset.idx2class[i].replace("_", " ") for i in range(test_dataset.num_classes)]
    text_in = {ModalityType.TEXT: ibdata.load_and_transform_text(class_names, device)}
    text_embs = F.normalize(backbone(text_in)[ModalityType.TEXT], dim=-1).cpu()

    audio_embs = F.normalize(test_embeddings, dim=-1).cpu()
    audio_labels = test_labels.cpu().numpy()

    sim_a2t = audio_embs @ text_embs.T
    a2t_recall = {}
    for k in ks:
        topk = sim_a2t.topk(k, dim=-1).indices.numpy()
        hits = np.array([audio_labels[i] in topk[i] for i in range(len(audio_labels))])
        a2t_recall[f"R@{k}"] = round(hits.mean() * 100, 2)

    sim_t2a = text_embs @ audio_embs.T
    t2a_recall = {}
    for k in ks:
        topk = sim_t2a.topk(k, dim=-1).indices.numpy()
        hits = []
        for c_idx in range(len(class_names)):
            true_audio_idx = np.where(audio_labels == c_idx)[0]
            hits.append(any(ai in topk[c_idx] for ai in true_audio_idx))
        t2a_recall[f"R@{k}"] = round(np.mean(hits) * 100, 2)

    return a2t_recall, t2a_recall


a2t_recall, t2a_recall = compute_recall_at_k(
    model, test_dataset, test_embeddings, test_labels, DEVICE,
)
print("Audio -> Text Recall@K:", a2t_recall)
print("Text -> Audio Recall@K:", t2a_recall)

with open("results/retrieval_recall.json", "w") as f:
    json.dump({"audio_to_text": a2t_recall, "text_to_audio": t2a_recall}, f, indent=2)


## 3.3 Tổng hợp kết quả

In [ ]:
import os, json
import pandas as pd

print('=' * 60)
print('  TONG HOP KET QUA — ImageBind CSC14005')
print('=' * 60)

with open('results/final_results.json') as f:
    res = json.load(f)

print(f'\n[Config]')
print(f'  Freeze mode : {res["freeze_mode"]}')
print(f'  Epochs      : {res["num_epochs"]}')

print(f'\n[Accuracy]')
print(f'  Best Val Top-1 : {res["best_val_acc"]:.2f}%')
print(f'  Test Top-1     : {res["test_top1"]:.2f}%')
print(f'  Test Top-5     : {res["test_top5"]:.2f}%')

with open('results/retrieval_recall.json') as f:
    ret = json.load(f)
print(f'\n[Cross-modal Retrieval]')
print(pd.DataFrame({'Audio→Text': ret['audio_to_text'], 'Text→Audio': ret['text_to_audio']}).to_string())

print(f'\n[Output files]')
for fn in sorted(os.listdir('results')):
    fp = os.path.join('results', fn)
    print(f'  {fn:45s} ({os.path.getsize(fp):,} bytes)')